# Synthetic Data Generator Pipeline
### Step 5 : Build Send Queue Stage


In [ ]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.send_queue_stage_writer import (
    build_sensor_messages_send_queue,
    build_sensor_messages_send_queue_sql_native,
    validate_sensor_messages_send_queue,
)

from utils.core.env_helpers import (
    env_int, 
    env_str,
)

In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

SEND_QUEUE_SOURCE_TABLE_NAME = env_str(
    "SYNTHETIC_SENSOR_MESSAGES_TABLE",
    "synthetic_sensor_messages_stage",
)

SEND_QUEUE_DESTINATION_TABLE_NAME = env_str(
    "SYNTHETIC_SEND_QUEUE_TABLE",
    "synthetic_sensor_messages_send_queue",
    aliases=("PRODUCER_QUEUE_TABLE",),
)

IF_EXISTS_FLAG = "replace"

QUEUE_STATUS_DEFAULT = "pending"
QUEUE_OWNER_ROLE = env_str("SYNTHETIC_QUEUE_OWNER_ROLE", "kafka_producer")
APPLY_OWNER_AND_GRANTS_FLAG = False
CHUNK_SIZE = env_int("SYNTHETIC_SEND_QUEUE_CHUNK_SIZE", 50000)

OBSERVATION_BATCH_SIZE = env_int("OBSERVATION_BATCH_SIZE", 500)

print("Synthetic send queue config")
print("schema:", SCHEMA)
print("source table:", SEND_QUEUE_SOURCE_TABLE_NAME)
print("target table:", SEND_QUEUE_DESTINATION_TABLE_NAME)
print("queue status default:", QUEUE_STATUS_DEFAULT)


Synthetic send queue config
schema: capstone
source table: synthetic_sensor_messages_stage
target table: synthetic_sensor_messages_send_queue
queue status default: pending
number of sensors: 52
observation batch size: 500
message batch size: 26000


---

In [14]:

engine = get_engine_from_env()

---

In [ ]:
NUMBER_OF_SENSORS = int(
    read_sql_dataframe(
        engine,
        f"""
        SELECT COUNT(DISTINCT sensor_index) AS sensor_count
        FROM "{SCHEMA}"."{SEND_QUEUE_SOURCE_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    ).loc[0, "sensor_count"]
)

PRODUCER_BATCH_SIZE = OBSERVATION_BATCH_SIZE * NUMBER_OF_SENSORS

print("Derived claim batch sizing")
print("number of sensors:", NUMBER_OF_SENSORS)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("producer batch size:", PRODUCER_BATCH_SIZE)

----

In [15]:
send_queue_table_name = build_sensor_messages_send_queue_sql_native(
    engine=engine,
    schema=SCHEMA,
    source_table=SEND_QUEUE_SOURCE_TABLE_NAME,
    target_table=SEND_QUEUE_DESTINATION_TABLE_NAME,
    if_exists=IF_EXISTS_FLAG,
    queue_status_default=QUEUE_STATUS_DEFAULT,
    queue_owner_role=QUEUE_OWNER_ROLE,
    apply_owner_and_grants=APPLY_OWNER_AND_GRANTS_FLAG,
    enable_timing_logging=True,
)

print("Built SQL-native send queue table:", send_queue_table_name)


[timing] source validation complete: 0.48 seconds
[timing] send queue table build complete: 43.06 seconds
[timing] runtime columns verified: 11.60 seconds
[timing] indexes/analyze complete: 205.21 seconds
Built SQL-native send queue table: synthetic_sensor_messages_send_queue


In [16]:
'''
send_queue_table_name = build_sensor_messages_send_queue(
    engine=engine,
    schema=SCHEMA,
    source_table=SEND_QUEUE_SOURCE_TABLE_NAME,
    target_table=SEND_QUEUE_DESTINATION_TABLE_NAME,
    if_exists=IF_EXISTS_FLAG,
    queue_status_default=QUEUE_STATUS_DEFAULT,
    chunk_size=CHUNK_SIZE,
    queue_owner_role=QUEUE_OWNER_ROLE,
    apply_owner_and_grants=APPLY_OWNER_AND_GRANTS_FLAG,
)

print("Built queue table:", send_queue_table_name)
'''

'\nsend_queue_table_name = build_sensor_messages_send_queue(\n    engine=engine,\n    schema=SCHEMA,\n    source_table=SEND_QUEUE_SOURCE_TABLE_NAME,\n    target_table=SEND_QUEUE_DESTINATION_TABLE_NAME,\n    if_exists=IF_EXISTS_FLAG,\n    queue_status_default=QUEUE_STATUS_DEFAULT,\n    chunk_size=CHUNK_SIZE,\n    queue_owner_role=QUEUE_OWNER_ROLE,\n    apply_owner_and_grants=APPLY_OWNER_AND_GRANTS_FLAG,\n)\n\nprint("Built queue table:", send_queue_table_name)\n'

---

In [17]:
validation_dataframe = validate_sensor_messages_send_queue(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_DESTINATION_TABLE_NAME,
)

display(validation_dataframe)

,row_count,distinct_observation_count,distinct_sensor_name_count,min_observation_timestamp,max_observation_timestamp,pending_count,unsent_count
0,11700000,225000,52,2026-04-16 00:00:00+00:00,2026-09-19 05:59:00+00:00,11700000,11700000


---

In [18]:
stage_05_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    WITH source_counts AS (
        SELECT
            COUNT(*) AS source_message_row_count
        FROM "{SCHEMA}"."{SEND_QUEUE_SOURCE_TABLE_NAME}"
    ),
    queue_counts AS (
        SELECT
            COUNT(*) AS queue_row_count,
            COUNT(DISTINCT message_key) AS distinct_message_key_count,
            COUNT(DISTINCT observation_index) AS distinct_observation_count,
            COUNT(DISTINCT sensor_name) AS distinct_sensor_name_count,
            COUNT(*) FILTER (WHERE queue_status = 'pending') AS pending_count,
            COUNT(*) FILTER (WHERE producer_sent_at IS NULL) AS unsent_count,
            COUNT(*) FILTER (WHERE claim_token IS NULL) AS unclaimed_count
        FROM "{SCHEMA}"."{SEND_QUEUE_DESTINATION_TABLE_NAME}"
    )
    SELECT
        source_counts.source_message_row_count,
        queue_counts.queue_row_count,
        queue_counts.distinct_message_key_count,
        queue_counts.distinct_observation_count,
        queue_counts.distinct_sensor_name_count,
        queue_counts.pending_count,
        queue_counts.unsent_count,
        queue_counts.unclaimed_count,
        queue_counts.queue_row_count - source_counts.source_message_row_count AS queue_row_delta,
        queue_counts.queue_row_count - queue_counts.distinct_message_key_count AS duplicate_message_key_count
    FROM source_counts
    CROSS JOIN queue_counts
    """
)

display(stage_05_validation_dataframe)

,source_message_row_count,queue_row_count,distinct_message_key_count,distinct_observation_count,distinct_sensor_name_count,pending_count,unsent_count,unclaimed_count,queue_row_delta,duplicate_message_key_count
0,11700000,11700000,11700000,225000,52,11700000,11700000,11700000,0,0


---

In [19]:
sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        message_key,
        observation_index,
        observation_timestamp,
        message_sequence_index,
        sensor_name,
        sensor_index,
        sensor_value,
        queue_status,
        claim_token,
        claimed_at,
        producer_topic,
        producer_worker_id,
        queued_at,
        producer_sent_at,
        producer_ack_at,
        producer_delivery_status,
        producer_delivery_error
    FROM {SCHEMA}.{SEND_QUEUE_DESTINATION_TABLE_NAME}
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 104
    """
)

display(sample_dataframe)

,message_key,observation_index,observation_timestamp,message_sequence_index,sensor_name,sensor_index,sensor_value,queue_status,claim_token,claimed_at,producer_topic,producer_worker_id,queued_at,producer_sent_at,producer_ack_at,producer_delivery_status,producer_delivery_error
0,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,2026-04-16 00:00:00+00:00,0,sensor_00,0,2.344329,pending,None,None,None,None,2026-05-26 00:29:51.868662+00:00,None,None,None,None
1,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,2026-04-16 00:00:00+00:00,1,sensor_01,1,48.327756,pending,None,None,None,None,2026-05-26 00:29:51.868662+00:00,None,None,None,None
2,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,2026-04-16 00:00:00+00:00,2,sensor_02,2,53.575950,pending,None,None,None,None,2026-05-26 00:29:51.868662+00:00,None,None,None,None
3,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,2026-04-16 00:00:00+00:00,3,sensor_03,3,43.424633,pending,None,None,None,None,2026-05-26 00:29:51.868662+00:00,None,None,None,None
4,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,2026-04-16 00:00:00+00:00,4,sensor_04,4,617.854335,pending,None,None,None,None,2026-05-26 00:29:51.868662+00:00,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,pump_synthetic_v1|synthetic_run_001|pump_asset...,2,2026-04-16 00:01:00+00:00,47,sensor_47,47,44.544907,pending,None,None,None,None,2026-05-26 00:29:51.868662+00:00,None,None,None,None
100,pump_synthetic_v1|synthetic_run_001|pump_asset...,2,2026-04-16 00:01:00+00:00,48,sensor_48,48,39.875294,pending,None,None,None,None,2026-05-26 00:29:51.868662+00:00,None,None,None,None
101,pump_synthetic_v1|synthetic_run_001|pump_asset...,2,2026-04-16 00:01:00+00:00,49,sensor_49,49,54.131620,pending,None,None,None,None,2026-05-26 00:29:51.868662+00:00,None,None,None,None
102,pump_synthetic_v1|synthetic_run_001|pump_asset...,2,2026-04-16 00:01:00+00:00,50,sensor_50,50,161.877550,pending,None,None,None,None,2026-05-26 00:29:51.868662+00:00,None,None,None,None


---

In [20]:
pending_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT *
    FROM capstone.synthetic_sensor_messages_send_queue
    WHERE queue_status = 'pending'
      AND producer_sent_at IS NULL
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 500
    """
)

display(pending_dataframe.head(100))

,dataset_id,run_id,asset_id,message_key,generated_row_id,observation_index,observation_timestamp,message_sequence_index,batch_id,row_in_batch,...,queue_status,queued_at,claim_token,claimed_at,producer_topic,producer_worker_id,producer_sent_at,producer_ack_at,producer_delivery_status,producer_delivery_error
0,pump_synthetic_v1,synthetic_run_001,pump_asset_001,pump_synthetic_v1|synthetic_run_001|pump_asset...,synthetic_run_001_obs_000000000001,1,2026-04-16 00:00:00+00:00,0,1,0,...,pending,2026-05-26 00:29:51.868662+00:00,None,None,None,None,None,None,None,None
1,pump_synthetic_v1,synthetic_run_001,pump_asset_001,pump_synthetic_v1|synthetic_run_001|pump_asset...,synthetic_run_001_obs_000000000001,1,2026-04-16 00:00:00+00:00,1,1,0,...,pending,2026-05-26 00:29:51.868662+00:00,None,None,None,None,None,None,None,None
2,pump_synthetic_v1,synthetic_run_001,pump_asset_001,pump_synthetic_v1|synthetic_run_001|pump_asset...,synthetic_run_001_obs_000000000001,1,2026-04-16 00:00:00+00:00,2,1,0,...,pending,2026-05-26 00:29:51.868662+00:00,None,None,None,None,None,None,None,None
3,pump_synthetic_v1,synthetic_run_001,pump_asset_001,pump_synthetic_v1|synthetic_run_001|pump_asset...,synthetic_run_001_obs_000000000001,1,2026-04-16 00:00:00+00:00,3,1,0,...,pending,2026-05-26 00:29:51.868662+00:00,None,None,None,None,None,None,None,None
4,pump_synthetic_v1,synthetic_run_001,pump_asset_001,pump_synthetic_v1|synthetic_run_001|pump_asset...,synthetic_run_001_obs_000000000001,1,2026-04-16 00:00:00+00:00,4,1,0,...,pending,2026-05-26 00:29:51.868662+00:00,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,pump_synthetic_v1,synthetic_run_001,pump_asset_001,pump_synthetic_v1|synthetic_run_001|pump_asset...,synthetic_run_001_obs_000000000002,2,2026-04-16 00:01:00+00:00,43,1,1,...,pending,2026-05-26 00:29:51.868662+00:00,None,None,None,None,None,None,None,None
96,pump_synthetic_v1,synthetic_run_001,pump_asset_001,pump_synthetic_v1|synthetic_run_001|pump_asset...,synthetic_run_001_obs_000000000002,2,2026-04-16 00:01:00+00:00,44,1,1,...,pending,2026-05-26 00:29:51.868662+00:00,None,None,None,None,None,None,None,None
97,pump_synthetic_v1,synthetic_run_001,pump_asset_001,pump_synthetic_v1|synthetic_run_001|pump_asset...,synthetic_run_001_obs_000000000002,2,2026-04-16 00:01:00+00:00,45,1,1,...,pending,2026-05-26 00:29:51.868662+00:00,None,None,None,None,None,None,None,None
98,pump_synthetic_v1,synthetic_run_001,pump_asset_001,pump_synthetic_v1|synthetic_run_001|pump_asset...,synthetic_run_001_obs_000000000002,2,2026-04-16 00:01:00+00:00,46,1,1,...,pending,2026-05-26 00:29:51.868662+00:00,None,None,None,None,None,None,None,None


In [21]:
queue_status_distribution_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count
    FROM "{SCHEMA}"."{SEND_QUEUE_DESTINATION_TABLE_NAME}"
    GROUP BY queue_status
    ORDER BY row_count DESC
    """
)

display(queue_status_distribution_dataframe)

,queue_status,row_count
0,pending,11700000


---

In [22]:
producer_pickup_preview_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        message_key,
        observation_index,
        message_sequence_index,
        sensor_index,
        sensor_name,
        sensor_value,
        queue_status,
        claim_token,
        producer_sent_at
    FROM "{SCHEMA}"."{SEND_QUEUE_DESTINATION_TABLE_NAME}"
    WHERE queue_status = 'pending'
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT {MESSAGE_BATCH_SIZE}
    """
)

display(producer_pickup_preview_dataframe.head(104))
print("Preview row count:", len(producer_pickup_preview_dataframe))

,message_key,observation_index,message_sequence_index,sensor_index,sensor_name,sensor_value,queue_status,claim_token,producer_sent_at
0,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,0,0,sensor_00,2.344329,pending,None,None
1,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,1,1,sensor_01,48.327756,pending,None,None
2,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,2,2,sensor_02,53.575950,pending,None,None
3,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,3,3,sensor_03,43.424633,pending,None,None
4,pump_synthetic_v1|synthetic_run_001|pump_asset...,1,4,4,sensor_04,617.854335,pending,None,None
...,...,...,...,...,...,...,...,...,...
99,pump_synthetic_v1|synthetic_run_001|pump_asset...,2,47,47,sensor_47,44.544907,pending,None,None
100,pump_synthetic_v1|synthetic_run_001|pump_asset...,2,48,48,sensor_48,39.875294,pending,None,None
101,pump_synthetic_v1|synthetic_run_001|pump_asset...,2,49,49,sensor_49,54.131620,pending,None,None
102,pump_synthetic_v1|synthetic_run_001|pump_asset...,2,50,50,sensor_50,161.877550,pending,None,None


Preview row count: 26000


In [23]:
ownership_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        schemaname,
        tablename,
        tableowner
    FROM pg_tables
    WHERE schemaname = :schema_name
      AND tablename = :table_name
    """,
    params={
        "schema_name": SCHEMA,
        "table_name": SEND_QUEUE_DESTINATION_TABLE_NAME,
    },
)

display(ownership_dataframe)

,schemaname,tablename,tableowner
0,capstone,synthetic_sensor_messages_send_queue,dcook_admin


---

In [24]:
# Dispose Engine
engine.dispose()